# Building a Tool-Calling Agent

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will build the canonical tool-calling agent loop: LLM reasons, calls tools, observes results, and repeats until it has a final answer.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define Math Tools

Use the `@tool` decorator to define tools. The docstring becomes the tool description the LLM sees.

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

tools = [add, multiply, divide]

## 4. Bind Tools to the LLM

`bind_tools()` attaches tool schemas to the model so it can generate structured tool calls.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

## 5. Build the Agent Graph

The graph has two nodes (LLM and tools) connected by a conditional edge that checks for tool calls.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def llm_node(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)

graph = StateGraph(MessagesState)

graph.add_node("llm", llm_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "llm")
graph.add_conditional_edges("llm", tools_condition)
graph.add_edge("tools", "llm")

app = graph.compile()

## 6. Visualize the Graph

In [ ]:
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

## 7. Run the Agent

Ask the agent a math question and watch it reason through tool calls.

In [ ]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [HumanMessage(content="What is 25 * 4 + 10?")]
})

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

## 8. Try More Questions

In [ ]:
questions = [
    "What is 100 / 4?",
    "What is 7 * 8 + 3 * 2?",
    "What is (50 + 25) * 2?",
]

for q in questions:
    result = app.invoke({"messages": [HumanMessage(content=q)]})
    answer = result["messages"][-1].content
    print(f"Q: {q}")
    print(f"A: {answer}\n")

## What You Learned

1. **`@tool`** decorator defines tools with type hints and docstrings
2. **`bind_tools()`** attaches tool schemas to the LLM
3. **`ToolNode`** executes tool calls and returns results as messages
4. **`tools_condition`** routes to the tool node or END based on whether tool calls exist
5. The agent **loops** until the LLM responds without requesting tool calls